# Reversible Transformer — Run 3b on Colab (reversible, max batch, fp32 stream)
**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine) → Save. Then **Runtime → Run all**.

The notebook: gets `colab_bundle.zip` (from Google Drive `MyDrive/colab_bundle.zip`, or asks you to upload it) →
runs the tests → finds the max batch on this GPU → trains **Run 3b** (50M tokens) → trains the same-GPU
**fp64 reference** → writes `REPORT_colab.md` → downloads `colab_results.zip` (and copies it to Drive if mounted).

In [ ]:
# 1. GPU + precision
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > GPU"
print(torch.__version__, torch.cuda.get_device_name())
DT = "bf16" if torch.cuda.is_bf16_supported() else "fp16"   # T4 -> fp16, L4/A100 -> bf16
RUN_REFERENCE = True   # also train the fp64-stream run on this GPU (fair tokens/s comparison); set False to skip
print("autocast dtype:", DT)

In [ ]:
# 2. Get the bundle: Google Drive first, otherwise upload from your PC
import os
os.chdir("/content")
ZIP = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.exists("/content/drive/MyDrive/colab_bundle.zip"):
        ZIP = "/content/drive/MyDrive/colab_bundle.zip"
except Exception as e:
    print("Drive not mounted:", e)
if ZIP is None:
    from google.colab import files
    print("Upload colab_bundle.zip (C:\sjk\erav5\disttraining2_pipelinepar\colab_bundle.zip)")
    ZIP = "/content/" + list(files.upload().keys())[0]
!unzip -o -q "$ZIP" -d /content/revtransformer
%cd /content/revtransformer
!ls -la . data

In [ ]:
# 3. Correctness tests (reversible grads == autograd, reconstruction, memory flat in depth)
!python -m pytest tests -q

In [ ]:
# 4. Max batch for the fp32-stream reversible model on THIS GPU, then Run 3b (50M tokens)
import json
!python find_max_batch.py --trunk reveuler_rev --h 0.5 --stream fp32 --start 64 --headroom 0.90 --dtype $DT --out results_colab
B = json.load(open("results_colab/maxbatch_reveuler_rev_fp32.json"))["max_batch"]
print("Run 3b batch =", B)
!python train.py --name run3b_rev_max_fp32stream --trunk reveuler_rev --h 0.5 --stream fp32 --batch $B --scale_lr --dtype $DT --out results_colab

In [ ]:
# 5. Same-GPU reference: fp64 stream at its own max batch (Run 3 settings)
if RUN_REFERENCE:
    !python find_max_batch.py --trunk reveuler_rev --h 0.5 --stream fp64 --start 64 --headroom 0.90 --dtype $DT --out results_colab
    B64 = json.load(open("results_colab/maxbatch_reveuler_rev_fp64.json"))["max_batch"]
    print("reference batch =", B64)
    !python train.py --name colab_run3_rev_max_fp64 --trunk reveuler_rev --h 0.5 --stream fp64 --batch $B64 --scale_lr --dtype $DT --out results_colab

In [ ]:
# 6. Report + download
!python colab_summary.py --results results_colab --write REPORT_colab.md
from IPython.display import Markdown, display
display(Markdown(open("REPORT_colab.md").read()))
!zip -q -r colab_results.zip results_colab REPORT_colab.md
if os.path.isdir("/content/drive/MyDrive"):
    !cp colab_results.zip /content/drive/MyDrive/
    print("copied to Google Drive: MyDrive/colab_results.zip")
from google.colab import files
files.download("colab_results.zip")